In [56]:
# # Import repo files
# !git clone -b hao https://github.com/sahitidoke/asym-model-simulation.git

In [57]:
# !cd /content/asym-model-simulation && git branch --show-current

In [58]:
# from google.colab import drive
# drive.mount('/content/drive')

In [59]:
import os
import sys
sys.path.append('/content/asym-model-simulation')
import contextlib
import subprocess
import time
import warnings
import numpy as np
import json
from datetime import datetime
from sklearn.covariance import graphical_lasso
from sklearn.exceptions import ConvergenceWarning
from matplotlib import pyplot as plt

from method import EM_algorithm as em, tlasso, stars
from simulation import simulation_data_generator as dg

COLOR_EM_DIAG = "#f5239a"
COLOR_EM_EXACT = "#2a78d6"
COLOR_GGM = "#e34948"
COLOR_T = "#f5a623"
COLOR_TS = "#23f57e"
COLOR_CHANCE = "#c3c2b7"

def log_step_ratio(rho_grid):
    """Multiplicative step of a log-spaced grid, so an extension stays even.

    A descending grid gives ratio < 1; multiplying the last rho by it
    repeatedly continues the same geometric sequence past the bottom of the
    grid, which is what the fp_end extension in roc_curve_em walks along.
    """
    g = np.asarray(rho_grid, dtype=float)
    if g.size < 2:
        raise ValueError("need at least two rho values to infer the step")
    ratio = float((g[-1] / g[0]) ** (1.0 / (g.size - 1)))
    if not 0.0 < ratio < 1.0:
        raise ValueError("rho grid must be strictly descending to be extended")
    return ratio

def edge_confusion(Theta_hat, true_pos_mask, true_neg_mask, tol=1e-8):
    iu = np.triu_indices(Theta_hat.shape[0], k=1)
    est_edges = np.abs(Theta_hat[iu]) > tol
    tp_rate = np.sum(est_edges & true_pos_mask) / true_pos_mask.sum()
    fp_rate = np.sum(est_edges & true_neg_mask) / true_neg_mask.sum()
    return fp_rate, tp_rate


@contextlib.contextmanager
def stream_to(path, also_stderr=True):
    """Send print()/warnings to `path` instead of the cell output.

    buffering=1 is the point: line buffering means the file fills as the run
    goes, so `Get-Content <path> -Wait` follows it live. On an exception the
    streams are restored before the traceback renders, so errors still show
    up in the notebook.
    """
    f = open(path, "w", buffering=1, encoding="utf-8")
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = f
    if also_stderr:
        sys.stderr = f
    try:
        yield f
    finally:
        sys.stdout, sys.stderr = old_out, old_err
        f.close()


@contextlib.contextmanager
def hush_convergence(counter):
    """Swallow graphical_lasso's ConvergenceWarning; count the fits that hit it.

    Every EM iteration of every method calls graphical_lasso, and it warns on
    most of them -- tens of thousands of identical lines per run, which buries
    everything else in the log.

    `counter` is a one-element list, incremented once per block that raised at
    least one ConvergenceWarning. Once, not once per warning: a single fit runs
    graphical_lasso every EM iteration and each of those emits one warning per
    inner coordinate-descent solve, so the warning count says nothing useful.
    Every other warning is re-emitted with its original file/line, so this
    silences the known noise and not warnings in general.
    """
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        yield
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        counter[0] += 1
    for w in caught:
        if not issubclass(w.category, ConvergenceWarning):
            warnings.warn_explicit(w.message, w.category, w.filename, w.lineno)


def roc_curve_em(
    Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
    algorithm_kwargs=None, rho_name="rho", theta_key="Theta",
    fp_end=None, max_extra=50, rho_floor=None,
):
    """Re-run the whole EM at each rho, warm-started, sparsest first.

    `algorithm_kwargs` holds whatever extra arguments the given algorithm takes
    (e.g. nu/n_iter for run_tlasso, n_burn/n_keep for run_em_MWGP); `rho_name`
    and `theta_key` cover algorithms that name the penalty or the returned
    precision matrix differently.

    Each fit is initialized at the previous rho's mu and Theta. Only those two
    are carried: nu/eta are tail-shape nuisance parameters, and handing them
    forward lets nu ratchet down across the sweep until lam = -2/nu - 0.5 is
    negative enough to overflow the Bessel terms in gig_moment.

    These EM objectives are not convex, so warm starting changes the estimator
    and not just the runtime: every point on the curve depends on the whole
    path prefix, and the sweep must stay sparse -> dense for the results to be
    reproducible.

    `fp_end` fixes where the curve ends instead of leaving it to rho_grid. A
    grid running to min_ratio * rho_max stops at whatever FPR that happens to
    land on, which differs by method and by replicate: some curves stop far
    short of the right edge (mean_roc then interpolates straight to (1, 1)
    across the gap), others spend most of their fits past the point where the
    curve is already saturated. With fp_end set the sweep ends at the first rho
    whose FPR reaches it, from either side:

      * reached inside rho_grid -> stop there, drop the rest of the grid (the
        early stop -- those fits would only add points beyond fp_end);
      * grid exhausted first -> keep stepping past its bottom end, same
        multiplicative step so the log spacing is unbroken, until fp_end is
        reached or `max_extra` extra fits have run.

    Either way the last point is the first one at FPR >= fp_end, so every curve
    ends at the same place. Extra points are warm started from the previous one
    exactly like the base grid, so the path stays a single sparse -> dense
    sweep.

    Returns (fp, tp, rho_used). rho_used is rho_grid extended or truncated to
    the rho values actually swept, so it -- not rho_grid -- is the grid the
    caller must keep.
    """
    kwargs = {} if algorithm_kwargs is None else dict(algorithm_kwargs)
    rhos = [float(r) for r in rho_grid]
    n_base = len(rhos)
    ratio = log_step_ratio(rhos) if fp_end is not None else None
    fp, tp = [], []
    prev = None
    n_nonconv = [0]
    n_stale = 0
    tail = f", extending until FPR >= {fp_end}" if fp_end is not None else ""
    print(f"Running {algorithm.__name__} over {n_base} rho values{tail}...")
    i = 0
    while i < len(rhos):
        extra = "" if i < n_base else f"  [extra {i - n_base + 1}/{max_extra}]"
        print(f"  rho={rhos[i]:.5f} ({i+1}/{len(rhos)}){extra}")

        t0 = time.perf_counter()
        with hush_convergence(n_nonconv):
            res = algorithm(
                Y, **{rho_name: rhos[i]}, **kwargs,
                **({"init": prev} if prev is not None else {}),
            )
        dt = time.perf_counter() - t0
        Theta_hat = res[theta_key]
        # A fit whose glasso raised is NOT a fit: run_em_diagonal/run_tlasso
        # keep the previous Theta and carry on, so the point recorded here
        # would describe a stale precision matrix. n_nu_clamped counts
        # coordinates driven to NU_MAX, which is how a degenerate theta_jj
        # announces itself.
        n_fail = int(res.get("n_glasso_fail", 0))
        n_clamp = int(res.get("n_nu_clamped", 0))
        n_stale += n_fail > 0
        if np.all(np.isfinite(Theta_hat)):
            prev = {"mu": res["mu"], "Theta": Theta_hat}

        fp_i, tp_i = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        fp.append(fp_i)
        tp.append(tp_i)
        flags = ""
        if n_fail:
            flags += f"  [STALE: glasso failed on {n_fail} EM iters]"
        if n_clamp:
            flags += f"  [nu->NU_MAX x{n_clamp}]"
        print(f"    fp={fp_i:.4f} tp={tp_i:.4f}  [{dt:.1f}s]{flags}")
        # Tail/skew estimates at this rho. run_tlasso and run_tstar_varlasso
        # return one scalar nu -- the value they were handed if it was held,
        # their ECM estimate if they were passed nu=None -- so it prints as
        # itself; run_em_diagonal's nu/eta are per coordinate and print as a
        # median. Which of the two a scalar is, and which parameterization the
        # per-coordinate ones use, is in the header at the top of the log.
        for _k in ("nu", "eta"):
            _v = res.get(_k)
            if _v is None:
                continue
            if np.ndim(_v) == 0:
                print(f"   {_k:3s} = {float(_v):.4f}")
            elif _k == "eta":
                # |eta|: with mixed-sign eta the signed median is ~0 by
                # construction, so it says nothing about how well eta is
                # recovered. The magnitude does.
                print(f"   median |{_k}| = {np.round(np.median(np.abs(_v)), 3)}")
            else:
                print(f"   median {_k:3s} = {np.round(np.median(_v), 3)}")
        i += 1

        if fp_end is not None and fp_i >= fp_end:
            # Early stop. Truncate rhos too, so it stays the grid that was
            # actually swept and lines up index-for-index with fp/tp.
            del rhos[i:]
            print(f"  FPR {fp_i:.4f} >= fp_end={fp_end}; stopping after "
                  f"{i} fits")
            break

        # Only ever extend off the current last point, so the added fits stay
        # in the same warm-started sparse -> dense order as the base grid.
        if (fp_end is not None and i == len(rhos)
                and i - n_base < max_extra):
            nxt = rhos[-1] * ratio
            # rho_floor is where the glasso subproblem stops being solvable.
            # S_tau has rank <= n < p, and the rho*I inside the glasso call is
            # the only thing conditioning it, so as rho -> 0 theta_jj blows up
            # (below ~1e-3 graphical_lasso raises outright). theta_jj then feeds
            # chi in the E-step and drives nu to NU_MAX. Ending the sweep is
            # honest; recording those points is not.
            if rho_floor is not None and nxt < rho_floor:
                print(f"  rho floor {rho_floor:g} reached at FPR {fp_i:.4f}; "
                      f"stopping (curve ends short of fp_end={fp_end})")
                break
            rhos.append(nxt)

    if n_nonconv[0]:
        print(f"  [note] {n_nonconv[0]}/{len(rhos)} fits hit a glasso "
              f"ConvergenceWarning (suppressed)")
    if n_stale:
        print(f"  [WARN] {n_stale}/{len(fp)} points came from a fit whose "
              f"glasso failed; those Thetas are stale, not estimates")
    if fp_end is not None and fp[-1] < fp_end:
        print(f"  [warn] FPR stalled at {fp[-1]:.4f} < fp_end={fp_end} after "
              f"{len(rhos) - n_base} extra fits (cap max_extra={max_extra}); "
              f"the curve stops short of fp_end")
    return np.asarray(fp), np.asarray(tp), np.asarray(rhos)


def roc_curve_em_autogrid(
    Y, algorithm, true_pos_mask, true_neg_mask, pilot_rho, n_rho,
    min_ratio=0.05, max_ratio=1, algorithm_kwargs=None, rho_name="rho",
    theta_key="Theta", S_key="S_tau", fp_end=None, max_extra=50,
    rho_floor=None,
):
    """pilot_rho_max -> make_rho_grid -> roc_curve_em, for one method/replicate.

    Returns (fp, tp, rho_grid); the grid is method- and replicate-specific and
    has to be kept, since the theoretical rho no longer sits at a common index.
    With fp_end set it is also variable-length -- n_rho is only the starting
    size, and roc_curve_em stops the sweep (or extends it below
    min_ratio * rho_max) at the first FPR >= fp_end -- so the returned grid is
    the one that was actually swept, not the one built here.

    The pilot fit is NOT reused as a warm start: the sweep must run
    sparse -> dense from its own top end for the path to be reproducible, and
    the pilot sits in the middle of it.
    """
    # The pilot is a full EM fit, so it warns like any other; keep it quiet
    # too. Its count is folded into the note the sweep prints below.
    n_nonconv = [0]
    with hush_convergence(n_nonconv):
        rho_max = stars.pilot_rho_max(
            Y, algorithm, pilot_rho, algorithm_kwargs=algorithm_kwargs,
            rho_name=rho_name, S_key=S_key,
        )
    if n_nonconv[0]:
        print("  [note] pilot fit hit a glasso ConvergenceWarning "
              "(suppressed)")
    if pilot_rho > rho_max:
        print(f"  [warn] pilot rho {pilot_rho:.5f} > rho_max {rho_max:.5f}: "
              f"it falls off the top of this method's grid")
    rho_grid = stars.make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp, rho_used = roc_curve_em(
        Y, rho_grid, algorithm, true_pos_mask, true_neg_mask,
        algorithm_kwargs=algorithm_kwargs, rho_name=rho_name,
        theta_key=theta_key, fp_end=fp_end, max_extra=max_extra,
        rho_floor=rho_floor,
    )
    if len(rho_used) > len(rho_grid):
        print(f"  grid extended by {len(rho_used) - len(rho_grid)} points "
              f"down to rho={rho_used[-1]:.5f}, ending at FPR {fp[-1]:.4f}")
    elif len(rho_used) < len(rho_grid):
        print(f"  grid stopped {len(rho_grid) - len(rho_used)} points early "
              f"at rho={rho_used[-1]:.5f}, ending at FPR {fp[-1]:.4f}")
    return fp, tp, rho_used


def roc_curve_glasso(Y, rho_grid, true_pos_mask, true_neg_mask,
                     glasso_kwargs=None, fp_end=None, max_extra=50,
                     rho_floor=None):
    """roc_curve_full_em analogue for the naive Gaussian glasso baseline.

    sklearn's graphical_lasso takes an empirical covariance rather than Y,
    names the penalty `alpha`, and returns (covariance, precision), so it needs
    its own loop. `fp_end` / `max_extra` end the sweep exactly as in
    roc_curve_em -- stop early or extend, whichever the FPR calls for -- and
    (fp, tp, rho_used) is returned for the same reason.
    """
    kwargs = {} if glasso_kwargs is None else dict(glasso_kwargs)
    S = np.cov(Y, rowvar=False) + 1e-10 * np.eye(Y.shape[1])
    p = Y.shape[1]
    rhos = [float(r) for r in rho_grid]
    n_base = len(rhos)
    ratio = log_step_ratio(rhos) if fp_end is not None else None
    fp, tp = [], []
    Theta_prev = np.eye(p)
    n_nonconv = [0]
    n_stale = 0
    tail = f", extending until FPR >= {fp_end}" if fp_end is not None else ""
    print(f"Running graphical_lasso over {n_base} rho values{tail}...")
    i = 0
    while i < len(rhos):
        extra = "" if i < n_base else f"  [extra {i - n_base + 1}/{max_extra}]"
        print(f"  rho={rhos[i]:.5f} ({i+1}/{len(rhos)}){extra}")
        t0 = time.perf_counter()
        with hush_convergence(n_nonconv):
            try:
                _, Theta_hat = graphical_lasso(S + rhos[i] * np.eye(p), alpha=rhos[i], **kwargs)
            except Exception as e:
                # Was silent. Reusing Theta_prev makes this rho a copy of the
                # previous one, not a fit, so it has to be visible -- the same
                # failure mode that hid run_em_diagonal's breakdown.
                print(f"    [WARN] glasso failed: {e}; reusing previous Theta")
                n_stale += 1
                Theta_hat = Theta_prev
        dt = time.perf_counter() - t0
        Theta_prev = Theta_hat
        fp_i, tp_i = edge_confusion(Theta_hat, true_pos_mask, true_neg_mask)
        fp.append(fp_i)
        tp.append(tp_i)
        print(f"    fp={fp_i:.4f} tp={tp_i:.4f}  [{dt:.1f}s]")
        i += 1

        if fp_end is not None and fp_i >= fp_end:
            del rhos[i:]
            print(f"  FPR {fp_i:.4f} >= fp_end={fp_end}; stopping after "
                  f"{i} fits")
            break

        if (fp_end is not None and i == len(rhos)
                and i - n_base < max_extra):
            nxt = rhos[-1] * ratio
            if rho_floor is not None and nxt < rho_floor:
                print(f"  rho floor {rho_floor:g} reached at FPR {fp_i:.4f}; "
                      f"stopping (curve ends short of fp_end={fp_end})")
                break
            rhos.append(nxt)

    if n_nonconv[0]:
        print(f"  [note] {n_nonconv[0]}/{len(rhos)} fits hit a glasso "
              f"ConvergenceWarning (suppressed)")
    if n_stale:
        print(f"  [WARN] {n_stale}/{len(fp)} points reused the previous Theta "
              f"after a glasso failure; those are not estimates")
    if fp_end is not None and fp[-1] < fp_end:
        print(f"  [warn] FPR stalled at {fp[-1]:.4f} < fp_end={fp_end} after "
              f"{len(rhos) - n_base} extra fits (cap max_extra={max_extra}); "
              f"the curve stops short of fp_end")
    return np.asarray(fp), np.asarray(tp), np.asarray(rhos)


def roc_curve_glasso_autogrid(
    Y, true_pos_mask, true_neg_mask, n_rho, min_ratio=0.05, max_ratio=1,
    glasso_kwargs=None, fp_end=None, max_extra=50, rho_floor=None,
):
    """roc_curve_em_autogrid for the Gaussian baseline; no pilot fit needed.

    glasso penalizes the empirical covariance itself, which does not depend on
    rho, so its rho_max is available in closed form -- the pilot fit would
    return the same S it was handed.
    """
    S = np.cov(Y, rowvar=False)
    rho_max = stars.offdiag_max(S)
    print(f"  graphical_lasso rho_max = max|S_ij| (off-diag) = {rho_max:.5f}")
    rho_grid = stars.make_rho_grid(
        rho_max, n_rho=n_rho, max_ratio=max_ratio, min_ratio=min_ratio,
    )
    print(f"  rho grid: {rho_grid}")
    fp, tp, rho_used = roc_curve_glasso(
        Y, rho_grid, true_pos_mask, true_neg_mask, glasso_kwargs=glasso_kwargs,
        fp_end=fp_end, max_extra=max_extra, rho_floor=rho_floor,
    )
    if len(rho_used) > len(rho_grid):
        print(f"  grid extended by {len(rho_used) - len(rho_grid)} points "
              f"down to rho={rho_used[-1]:.5f}, ending at FPR {fp[-1]:.4f}")
    elif len(rho_used) < len(rho_grid):
        print(f"  grid stopped {len(rho_grid) - len(rho_used)} points early "
              f"at rho={rho_used[-1]:.5f}, ending at FPR {fp[-1]:.4f}")
    return fp, tp, rho_used


def auc_from_curve(fp, tp, fp_end=None):
    """Area under one replicate's ROC curve, over [0, fp_end] exactly.

    The sweep stops at the FIRST rho whose FPR reaches fp_end, so the last
    point overshoots it by a different amount in every replicate and every
    method -- 0.51 here, 0.58 there. Integrating out to wherever it happened
    to land rewards the ones that overshot most with extra area, which has
    nothing to do with how well they recovered the support. So the curve is
    cut at exactly fp_end: the TPR there is linearly interpolated between the
    two fits that straddle it, everything past it is dropped, and every AUC is
    then an integral over the same interval.

    Anchored at (0, 0) as in mean_roc -- rho at the top of the grid zeroes
    every off-diagonal, so that is a real point of the path, and the integral
    needs the curve defined from 0. Ties in FPR keep the largest TPR (the
    upper envelope), again as in mean_roc, so this is the area under the same
    curve that mean_roc interpolates.

    The result is a PARTIAL area: it is bounded by fp_end, not by 1, and is
    not rescaled, so these AUCs compare to each other only at a common fp_end.
    fp_end=None integrates over whatever range the curve covers.
    """
    x = np.concatenate([[0.0], np.asarray(fp, dtype=float)])
    y = np.concatenate([[0.0], np.asarray(tp, dtype=float)])
    order = np.argsort(x, kind="stable")
    x, y = x[order], y[order]
    x_u, first = np.unique(x, return_index=True)
    y_u = np.maximum.reduceat(y, first)
    if fp_end is not None:
        fp_end = float(fp_end)
        if x_u[-1] < fp_end:
            # Only when the sweep stalled below fp_end (max_extra ran out).
            # np.interp holds the last TPR flat out to fp_end rather than
            # extrapolating; flag it, since that stretch was never fit.
            print(f"  [warn] AUC integrates to fp_end={fp_end} but the curve "
                  f"stops at FPR {x_u[-1]:.4f}; TPR held flat above that")
        y_at_end = np.interp(fp_end, x_u, y_u)
        keep = x_u < fp_end
        x_u = np.append(x_u[keep], fp_end)
        y_u = np.append(y_u[keep], y_at_end)
    return np.trapezoid(y_u, x_u)


def mean_roc(fp, tp, n_points=201, fp_end=None):
    """Vertical average of the replicate ROC curves (Fawcett 2006, Sec. 6.1).

    Averaging fp and tp down a column would be threshold averaging, which needs
    column i to be the same threshold in every row. It is not: every replicate
    has its own rho grid built from its own rho_max, so column i is a different
    rho in every row. Averaging it smears the curve horizontally, and where the
    curves are convex the mean point can land below all of them.

    Interpolating each replicate's TPR onto a common FPR grid and averaging
    vertically needs no correspondence between the rho grids at all. Returns
    (fp_common, tp_mean, tp_se).

    `fp` and `tp` are sequences of per-replicate curves, not necessarily a
    rectangular array: with fp_end set, each replicate takes however many fits
    it needs to reach that FPR, so the curves differ in length (they all end at
    the same FPR, not after the same number of points).

    The averaged curve stops where the sweeps stopped -- it is NOT anchored at
    (1, 1). That anchor would draw a straight line from the last fitted point
    to the top-right corner and present it as estimated, when nothing was fit
    out there; the sweep ended at fp_end deliberately. So `fp_common` runs
    [0, fp_end], or, if fp_end is not given, up to the smallest FPR every
    replicate reached -- the widest range all of them actually cover, over
    which np.interp never extrapolates. (0, 0) stays: rho at the top of the
    grid zeroes every off-diagonal, so an empty graph is a real point of the
    path rather than a drawn-in corner.
    """
    fp = [np.asarray(f, dtype=float) for f in fp]
    tp = [np.asarray(t, dtype=float) for t in tp]
    n_rep = len(fp)
    reached = min(float(f.max()) for f in fp)
    fp_hi = reached if fp_end is None else float(fp_end)
    if fp_hi > reached:
        print(f"  [warn] mean_roc averages out to FPR {fp_hi:.3f} but the "
              f"shortest replicate only reached {reached:.3f}; its TPR is held "
              f"flat above that")
    fp_common = np.linspace(0.0, fp_hi, n_points)
    tp_interp = np.empty((n_rep, n_points))
    for r in range(n_rep):
        # Anchor at (0, 0) only -- see the docstring on why (1, 1) is not one.
        x = np.concatenate([[0.0], fp[r]])
        y = np.concatenate([[0.0], tp[r]])
        order = np.argsort(x, kind="stable")
        x, y = x[order], y[order]
        # np.interp needs strictly increasing x. Ties are one FPR reached at
        # several rho; keep the best TPR there, i.e. the upper envelope.
        x_u, first = np.unique(x, return_index=True)
        y_u = np.maximum.reduceat(y, first)
        tp_interp[r] = np.interp(fp_common, x_u, y_u)
    tp_se = (
        tp_interp.std(axis=0, ddof=1) / np.sqrt(n_rep) if n_rep > 1
        else np.zeros(n_points)
    )
    return fp_common, tp_interp.mean(axis=0), tp_se


def git_commit():
    """Short HEAD of the checkout the methods came from, flagged dirty.

    The log records which code produced it: `filename` says what the run was
    for, this says what actually ran. Resolved from the `method` package rather
    than the cwd, so it reports the repo under test wherever the notebook is
    being run from -- including a Colab clone. "unknown" if git is unavailable.
    """
    repo = os.path.dirname(os.path.dirname(os.path.abspath(em.__file__)))
    try:
        head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo,
                              capture_output=True, text=True, timeout=5).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain"], cwd=repo,
                               capture_output=True, text=True, timeout=5).stdout.strip()
        return f"{head or 'unknown'}{' (dirty working tree)' if dirty else ''}"
    except Exception:
        return "unknown"


def describe_nu(kwargs):
    """How a run_tlasso / run_tstar_varlasso fit is treating nu, in words.

    Those two take nu=None to mean "estimate it" and a number to mean "hold it
    there", so their kwargs dict is the whole story -- which is the point of
    reading it here rather than restating it by hand in the header.
    """
    if "nu" not in kwargs:
        return "FIXED at the function default (nu = 3.0), not estimated"
    nu = kwargs["nu"]
    if nu is None:
        return ("ESTIMATED by the ECM step, bracket "
                f"[{tlasso.NU_MIN}, {tlasso.NU_MAX}]; restarted from 3.0 at "
                "every rho, since the warm start carries only mu and Theta")
    return f"FIXED at nu = {float(nu):.6g}, not estimated"


def describe_nu_em_diag(kwargs):
    """Same, for run_em_diagonal, which names the switch nu_fixed and estimates
    a whole vector.

    The parameterization is inverted relative to tlasso's and the two nu's land
    in the same log, so the header says so outright: EM_algorithm's tau_j is
    Inv-Gamma(2/nu_j, 2/nu_j), where LARGE nu_j is the heavy tail, while
    tlasso's tau ~ Gamma(nu/2, nu/2) makes SMALL nu the heavy tail.
    """
    nu_fixed = kwargs.get("nu_fixed")
    if nu_fixed is None:
        return (f"ESTIMATED per coordinate (nu_j), bracket "
                f"[{em.NU_MIN}, {em.NU_MAX}], nu_j = 0 meaning that coordinate "
                f"went Gaussian")
    return f"FIXED at nu_j = {float(nu_fixed):.6g} for every coordinate"


def describe_vec(name, v, width=8):
    """Summary line for a true-parameter vector: constant ones say so, the rest
    report min/median/max. format_vec prints the entries themselves under it."""
    v = np.asarray(v, dtype=float)
    if np.allclose(v, v.flat[0]):
        return f"{name:<{width}}: all {v.flat[0]:.4g}   (length {v.size})"
    return (f"{name:<{width}}: min {v.min():.4g}   median {np.median(v):.4g}   "
            f"max {v.max():.4g}   (length {v.size})")


def format_vec(v, indent=" " * 12, precision=4, width=78):
    """Every entry of a true-parameter vector, wrapped and indented.

    threshold=inf, so numpy never elides the middle into "..." -- the point of
    this block is that the log holds the actual draw. The min/median/max line
    from describe_vec stays above it as the summary; this is the record, and it
    is what lets a finished run be tied to the exact eta/nu it was fit against
    without the notebook state.
    """
    body = np.array2string(np.asarray(v, dtype=float), precision=precision,
                           separator=", ", max_line_width=width - len(indent),
                           threshold=np.inf, floatmode="fixed")
    return "\n".join(indent + line for line in body.splitlines())


def run_header(out_dir, filename, methods):
    """Title block for a run's log: what was simulated, and how each method was
    told to treat nu.

    Called from inside stream_to, so it lands in that run's own log above the
    fits it describes. Everything is read at call time from the globals the run
    itself uses -- there is nothing to keep in sync by hand, and a log found
    months later says which run produced it without the notebook state.

    `methods` is a sequence of (label, kwargs, nu_description); build the last
    with describe_nu / describe_nu_em_diag.
    """
    bar = "=" * 78
    print(bar)
    print(f"ROC run: {filename}")
    print(bar)
    print(f"  started : {datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"  outputs : {out_dir}/   (per-replicate figures, JSON, this log)")
    print(f"  commit  : {git_commit()}")

    print("\n[design]")
    print(f"  replicates       : {num_simulations}")
    print(f"  p, n             : {p}, {n}")
    print(f"  rho grid         : {num_rho} points, min_ratio 0.01, rebuilt per "
          f"method and replicate off that method's own converged S_tau")
    print(f"  theoretical rho  : sqrt(log p / n) = {theoretical_rho:.5f}  "
          f"(pilot only; not a point on the curve)")
    print(f"  FP_end, max_extra: {FP_end}, {max_extra}")

    print("\n[true parameters]  simulate_aat_data")
    print(f"  {describe_vec('mu_true', mu_true)}")
    print(f"  {describe_vec('eta_true', eta_true)}")
    print(format_vec(eta_true))
    print(f"  {describe_vec('nu_true', nu_true)}")
    print(format_vec(nu_true))
    n_edge, n_pair = int(true_pos_mask.sum()), int(true_pos_mask.size)
    ev = np.linalg.eigvalsh(Theta_true)
    print(f"  Theta   : {n_edge}/{n_pair} edges ({n_edge / n_pair:.2%} of pairs), "
          f"eigenvalues [{ev.min():.3f}, {ev.max():.3f}], "
          f"max |theta_jk| off-diagonal {np.abs(Theta_true[iu]).max():.3f}")
    print("            NOTE: this is the SUPPORT of Theta, which is the target "
          "of the ROC curves;")
    print("            it is not the conditional independence graph of Y in "
          "this model.")
    ent = getattr(getattr(rng.bit_generator, "seed_seq", None), "entropy", None)
    print(f"  rng     : entropy {ent}")
    print(f"            (np.random.default_rng({ent}) replays the same draws)")

    print("\n[methods]  nu treatment for this run")
    for label, kwargs, nu_desc in methods:
        print(f"  {label}")
        print(f"      nu   : {nu_desc}")
        rest = ", ".join(f"{k}={v!r}" for k, v in kwargs.items()
                         if k not in ("nu", "nu_fixed"))
        print(f"      args : {rest or '(defaults)'}")
    print("  Reading the per-rho nu lines below: tlasso and t*var-lasso print a "
          "scalar `nu`,")
    print("  run_em_diagonal a `median nu`/`median eta` over coordinates. The "
          "two nu's are NOT")
    print("  comparable -- tau ~ Gamma(nu/2, nu/2) for the t models, so small "
          "nu is the heavy")
    print("  tail, while EM_algorithm's tau_j ~ Inv-Gamma(2/nu_j, 2/nu_j) puts "
          "the heavy tail at")
    print("  large nu_j.")
    print(bar + "\n")


In [60]:
filename = "alternate_nu_eta_all_nu_variable"
num_simulations=50
p=100
n=50
num_rho=20
# Where every ROC curve ends: the sweep stops at the first rho whose FPR
# reaches FP_end, extending below min_ratio * rho_max (at most max_extra extra
# fits) if the grid runs out first. So the curves all cover the same FPR range
# instead of ending wherever the fixed grid happened to land, and no fits are
# spent past FP_end. FP_end = None restores the old fixed-length grids.
FP_end = 0.5
max_extra = 50
MIN_RATIO = 0.01

# One folder per run, named after `filename`: the per-replicate figures, the
# JSON dump and the log all go in it. Runs no longer overwrite each other's
# outputs (or share one roc_run.log that the next run truncates), and a folder
# carries everything needed to read its own results.
RUN_DIR = os.path.join("results", "simulations", "roc", filename)
os.makedirs(RUN_DIR, exist_ok=True)
LOG_PATH = os.path.join(RUN_DIR, f"{filename}.log")

In [61]:
# rng = np.random.default_rng()
# mu_true  = np.zeros(p)
# eta_true = np.full(p,4)
# nu_true = np.full(p,0.5)
# Theta_true = dg.make_true_theta(p)

# iu = np.triu_indices(p, k=1)
# theoretical_rho = np.sqrt(np.log(p) / n)
# true_pos_mask = Theta_true[iu] != 0
# true_neg_mask = ~true_pos_mask

In [62]:
# True parameters are fixed by hand and seeded, not redrawn per run: drawing
# eta and nu from wide uniforms made every run a different problem, which is
# what made the gap over t*var-lasso move around between runs.
#
# tau_j ~ Inv-Gamma(2/nu_j, 2/nu_j), so coordinate j has df_j = 4/nu_j, and
# E[tau_j^k] is finite only for k < 2/nu_j. So skew(Y_j) exists at all only
# when nu_j < 2/3; both levels below stay well inside that.
SEED = 20260814
rng = np.random.default_rng(SEED)

mu_true = np.zeros(p)
Theta_true = dg.make_true_theta(p, rng=rng)

# nu first: two levels, alternating, 50/50 across coordinates.
nu_true = np.where(np.arange(p) % 2 == 0, 0.25, 0.50)      # df 16 and df 8

# eta second, one value per nu level, signs mixed. The skewness of Y_j grows
# with eta_j*nu_j but saturates, and it saturates far lower at a light tail
# (the ceiling is 1.96 at nu=0.25 against 5.66 at nu=0.50), so the light half
# needs the much larger eta to be just as skewed. These two values put every
# coordinate at |skew(Y_j)| between 0.56 and 1.04 -- moderately skewed
# everywhere, with none left effectively symmetric.
eta_true = rng.choice([-1.0, 1.0], size=p) * np.where(nu_true == 0.25, 8, 3)

iu = np.triu_indices(p, k=1)
theoretical_rho = np.sqrt(np.log(p) / n)
true_pos_mask = Theta_true[iu] != 0
true_neg_mask = ~true_pos_mask

In [63]:
# The single scalar nu handed to tlasso and t*var-lasso, set close to the truth
# so neither is handicapped by a bad tuning value: the comparison should be
# about the asymmetry the two symmetric t-models cannot represent, not about
# the df they were given.
#
# Their tau ~ Gamma(nu/2, nu/2) parameterizes nu AS the degrees of freedom,
# while ours has df_j = 4/nu_j, so the conversion happens per coordinate and
# only then is summarized -- median(4/nu_true), not 4/median(nu_true). With
# true df of 16 and 8 that gives 12, the midpoint, rather than 10.67.
NU_tlasso = float(np.median(4.0 / nu_true))
print(f"Fixed scalar nu (= df) for tlasso and t*_var-lasso: {NU_tlasso}")
print(f"  true per-coordinate df: {np.unique(4.0 / nu_true)}")

Fixed scalar nu (= df) for tlasso and t*_var-lasso: 12.0
  true per-coordinate df: [ 8. 16.]


In [64]:
# Curves are ragged now: with FP_end set, every replicate ends at the same FPR
# but takes its own number of fits to get there, so these are lists of 1D
# arrays rather than (num_simulations, num_rho) blocks.
# mean_roc and the [:k] slicing in save_curve take them as such.
fp_em_diag, tp_em_diag = [], []
fp_em_exact, tp_em_exact = [], []
fp_ggm, tp_ggm = [], []
fp_t, tp_t = [], []
fp_ts, tp_ts = [], []
auc_em_diag = np.zeros(num_simulations)
auc_em_exact = np.zeros(num_simulations)
auc_ggm = np.zeros(num_simulations)
auc_t = np.zeros(num_simulations)
auc_ts = np.zeros(num_simulations)
# One grid per method per replicate: rho_max is read off that method's own
# converged S_tau, so index i is the same relative position on the path
# (min_ratio ... 1 of rho_max) but not the same rho across methods.
rho_grids_em_diag = []
rho_grids_em_exact = []
rho_grids_ggm = []
rho_grids_t = []
rho_grids_ts = []

In [65]:
def save_curve(sim, filename=None):
    """Rebuild the figure from the first `sim` replicates already in memory.

    Nothing is refit: every stored curve list is sliced to [:sim]. mean_roc is
    a vertical average over replicates and each AUC was computed from its own
    replicate's curve, so a prefix of them is exactly the run you would have
    got had the loop stopped at `sim`. The fp/tp entries are ragged (FP_end ends
    every replicate at the same FPR, not after the same number of fits), which
    is why the slicing is all that happens here -- nothing stacks them.
    """
    k = int(sim)
    if not 1 <= k <= num_simulations:
        raise ValueError(f"sim must be in 1..{num_simulations}, got {sim}")

    def se(a):
        # ddof=1 is undefined for one replicate; report 0 rather than nan.
        return a.std(ddof=1) / np.sqrt(k) if k > 1 else 0.0

    fp_em_diag_k,  tp_em_diag_k  = fp_em_diag[:k],  tp_em_diag[:k]
    fp_em_exact_k, tp_em_exact_k = fp_em_exact[:k], tp_em_exact[:k]
    fp_ggm_k,      tp_ggm_k      = fp_ggm[:k],      tp_ggm[:k]
    fp_t_k,        tp_t_k        = fp_t[:k],        tp_t[:k]
    fp_ts_k,       tp_ts_k       = fp_ts[:k],       tp_ts[:k]

    auc_em_diag_k  = auc_em_diag[:k]
    auc_em_exact_k = auc_em_exact[:k]
    auc_ggm_k     = auc_ggm[:k]
    auc_t_k       = auc_t[:k]
    auc_ts_k      = auc_ts[:k]

    grids_em_diag_k  = rho_grids_em_diag[:k]
    grids_em_exact_k = rho_grids_em_exact[:k]
    grids_ggm_k     = rho_grids_ggm[:k]
    grids_t_k       = rho_grids_t[:k]
    grids_ts_k      = rho_grids_ts[:k]

    # fp_end here, not None: every method stopped at that FPR, so it is the
    # common right edge. Left to itself mean_roc would pick the smallest FPR
    # reached, which is a hair past fp_end and differs per method.
    fp_common, tp_em_diag_mean, tp_em_diag_se = mean_roc(fp_em_diag_k, tp_em_diag_k, fp_end=FP_end)
    _, tp_em_exact_mean, tp_em_exact_se = mean_roc(fp_em_exact_k, tp_em_exact_k, fp_end=FP_end)
    _, tp_ggm_mean, tp_ggm_se = mean_roc(fp_ggm_k, tp_ggm_k, fp_end=FP_end)
    _, tp_t_mean, tp_t_se = mean_roc(fp_t_k, tp_t_k, fp_end=FP_end)
    _, tp_ts_mean, tp_ts_se = mean_roc(fp_ts_k, tp_ts_k, fp_end=FP_end)

    print(f"\n(p={p}, n={n}) over {k} replicates")
    print(f"  Asymmetric model (Diagonal): AUC = {auc_em_diag_k.mean():.3f} "
          f"(SE {se(auc_em_diag_k):.3f})")
    print(f"  Asymmetric model (Exact):    AUC = {auc_em_exact_k.mean():.3f} "
          f"(SE {se(auc_em_exact_k):.3f})")
    print(f"  Classical t-model (TLASSO): AUC = {auc_t_k.mean():.3f} "
          f"(SE {se(auc_t_k):.3f})")
    print(f"  Alternative t-model (TSTAR_VARLASSO): AUC = {auc_ts_k.mean():.3f} "
          f"(SE {se(auc_ts_k):.3f})")
    print(f"  Naive Gaussian glasso (GGM): AUC = {auc_ggm_k.mean():.3f} "
          f"(SE {se(auc_ggm_k):.3f})")

    METHODS = (
        ("Asym. diag. model", fp_em_diag_k, tp_em_diag_k, tp_em_diag_mean,
         tp_em_diag_se, auc_em_diag_k, grids_em_diag_k, COLOR_EM_DIAG, "-"),
        ("Asym. exact model", fp_em_exact_k, tp_em_exact_k, tp_em_exact_mean,
         tp_em_exact_se, auc_em_exact_k, grids_em_exact_k, COLOR_EM_EXACT, "-"),
        ("Naive GGM", fp_ggm_k, tp_ggm_k, tp_ggm_mean, tp_ggm_se, auc_ggm_k,
         grids_ggm_k, COLOR_GGM, "-."),
        ("Classical t-model", fp_t_k, tp_t_k, tp_t_mean, tp_t_se, auc_t_k,
         grids_t_k, COLOR_T, "-."),
        ("Alternative t-model", fp_ts_k, tp_ts_k, tp_ts_mean, tp_ts_se,
         auc_ts_k, grids_ts_k, COLOR_TS, "-."),
    )

    fp_hi = float(fp_common[-1])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, fp_hi], [0, fp_hi], linestyle="--", linewidth=1, color=COLOR_CHANCE)
    for name, fp_m, tp_m, tp_mean, tp_se, auc, grids, color, ls in METHODS:
        ax.plot(fp_common, tp_mean, color=color, linewidth=2, linestyle=ls,
                label=f"{name}, avg AUC={auc.mean():.3f}")
        ax.fill_between(fp_common, tp_mean - tp_se, tp_mean + tp_se,
                        color=color, alpha=0.15, linewidth=0)
        # The theoretical rho is a threshold, not a position on the averaged
        # curve: take it in each replicate and average those operating points.
        # It can sit slightly off the mean curve, which is honest -- that curve
        # is a vertical average and this point is not.
    ax.plot([], [], "o", color="#52514e",
            label=r"$\rho=\sqrt{\log p\,/\,n}$" + f" = {theoretical_rho:.3g}")

    # The curves end at FP_end; showing empty space out to FPR = 1 would only
    # invite reading the gap as part of the estimate.
    ax.set_xlim(0, fp_hi)
    ax.set_ylim(0, 1.2)
    ax.set_xlabel("false positive rate (1 - specificity)")
    ax.set_ylabel("true positive rate (sensitivity)")
    ax.set_title(f"Precision-matrix support recovery: p={p}, n={n}")
    ax.legend(loc="lower right")
    fig.tight_layout()

    out = os.path.join(RUN_DIR, f"roc_{k}" if filename is None
                                else f"{filename}_{k}")
    fig.savefig(f"{out}.pdf")

In [ ]:
# Output goes to this run's own log, RUN_DIR/<filename>.log, so it does not
# flood the cell and does not get truncated by the next run.
# Follow it live:  Get-Content $LOG_PATH -Wait -Tail 20
# The log opens with a header block (run_header) recording the true parameters,
# how each method is treating nu, and the rest of the run's settings; the sweep
# then prints nu at every rho for both t-methods.
#
# Each method gets its own rho grid in each replicate: fit at the theoretical
# rho first, let it converge, then take rho_max = max_{i != j} |S_ij| off the
# converged S_tau. See pilot_rho_max for why a shared cov(Y) grid is wrong here.
#
# num_rho is only the starting length: each sweep ends at the first rho with
# FPR >= FP_end, stopping early or extending below the grid (at most max_extra
# extra fits) to get there, so the stored curves and grids are ragged. See
# roc_curve_em.
TOL = 1e-5
kwargs_t = {"n_iter": 200, "verbose": False, "tol": TOL, "nu": None}
kwargs_ts = {"n_iter": 200, "verbose": False, "tol": TOL, "nu": None}
kwargs_em_diag = {"n_iter": 200, "verbose": False, "tol": TOL}
# run_em_exact solves the full (mu, gamma) system against the whole Theta
# instead of its diagonal. It has no nu_fixed switch -- nu is always estimated
# per coordinate -- so describe_nu_em_diag reports it as such below.
kwargs_em_exact = {"n_iter": 200, "verbose": False, "tol": TOL}
# Bound once and passed to both the header and the fit below, so the header
# cannot describe a run different from the one that executes.
kwargs_ggm = {"max_iter": 2000, "verbose": False}

with stream_to(LOG_PATH):
    run_header(RUN_DIR, filename, [
        ("Classical t         (run_tlasso)", kwargs_t, describe_nu(kwargs_t)),
        ("Alternative t*      (run_tstar_varlasso)", kwargs_ts, describe_nu(kwargs_ts)),
        ("Asymmetric alt-t    (run_em_diagonal)", kwargs_em_diag,
         describe_nu_em_diag(kwargs_em_diag)),
        ("Asymmetric alt-t    (run_em_exact)", kwargs_em_exact,
         describe_nu_em_diag(kwargs_em_exact)),
        ("Naive Gaussian      (graphical_lasso)", kwargs_ggm,
         "not a parameter of this model"),
    ])
    for sim in range(num_simulations):
        print(f"Replicate {sim + 1}/{num_simulations}")

        # Generate data from an independent model (noisy skewed Gaussian)
        Y,_ = dg.simulate_aat_data(n, p, Theta_true, mu_true, eta_true, nu_true, rng)

        # Classical t-distribution model (run_tlasso)
        fp, tp, grid = roc_curve_em_autogrid(
            Y, tlasso.run_tlasso, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=MIN_RATIO,
            algorithm_kwargs=kwargs_t, fp_end=FP_end, max_extra=max_extra,
        )
        fp_t.append(fp); tp_t.append(tp); rho_grids_t.append(grid)
        auc_t[sim] = auc_from_curve(fp, tp, fp_end=FP_end)

        # Alternative t-distribution model (run_tstar_varlasso)
        fp, tp, grid = roc_curve_em_autogrid(
            Y, tlasso.run_tstar_varlasso, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=MIN_RATIO,
            algorithm_kwargs=kwargs_ts, fp_end=FP_end, max_extra=max_extra,
        )
        fp_ts.append(fp); tp_ts.append(tp); rho_grids_ts.append(grid)
        auc_ts[sim] = auc_from_curve(fp, tp, fp_end=FP_end)

        # Asymmetric Alternative t-distribution model (EM_DIAGONAL)
        fp, tp, grid = roc_curve_em_autogrid(
            Y, em.run_em_diagonal, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=MIN_RATIO,
            algorithm_kwargs=kwargs_em_diag, fp_end=FP_end, max_extra=max_extra,
        )
        fp_em_diag.append(fp); tp_em_diag.append(tp); rho_grids_em_diag.append(grid)
        auc_em_diag[sim] = auc_from_curve(fp, tp, fp_end=FP_end)

        # Asymmetric Alternative t-distribution model (EM_EXACT)
        fp, tp, grid = roc_curve_em_autogrid(
            Y, em.run_em_exact, true_pos_mask, true_neg_mask,
            pilot_rho=theoretical_rho, n_rho=num_rho, min_ratio=MIN_RATIO,
            algorithm_kwargs=kwargs_em_exact, fp_end=FP_end, max_extra=max_extra,
        )
        fp_em_exact.append(fp); tp_em_exact.append(tp); rho_grids_em_exact.append(grid)
        auc_em_exact[sim] = auc_from_curve(fp, tp, fp_end=FP_end)

        # Naive Gaussian graphical lasso baseline
        fp, tp, grid = roc_curve_glasso_autogrid(
            Y, true_pos_mask, true_neg_mask, n_rho=num_rho, min_ratio=MIN_RATIO,
            glasso_kwargs=kwargs_ggm,
            fp_end=FP_end, max_extra=max_extra,
        )
        fp_ggm.append(fp); tp_ggm.append(tp); rho_grids_ggm.append(grid)
        auc_ggm[sim] = auc_from_curve(fp, tp, fp_end=FP_end)

        save_curve(sim+1, filename = filename)